Step 1 Import everything we need

In [ ]:
import os
from dotenv import load_dotenv #to acess the API key seceretely

# libraires for langchain
from langchain_community.document_loaders import TextLoader
# splitting data
from langchain_text_splitters import RecursiveCharacterTextSplitter

#embeddings
from langchain_community.embeddings import JinaEmbeddings

# vector DB
from langchain_community.vectorstores import FAISS

#LLM
from langchain_groq import ChatGroq

#AI AGENT
from langchain.agents import create_agent




In [63]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [64]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


LOADING OUR DATA

In [65]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [66]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content="COGNIVEXA AI SOLUTIONS PVT. LTD.\nHUMAN RESOURCES POLICY\n\nDocument Title: Employee Human Resources Policy\nCompany: CognivexaAI Solutions Pvt. Ltd.\nDocument Version: 1.0\nEffective Date: 07 August 2026\nDocument Owner: Human Resources Department\n\n\n1. COMPANY OVERVIEW\n\nCognivexaAI Solutions Pvt. Ltd. is a technology company focused on Artificial Intelligence, Machine Learning, Generative AI, software engineering, data analytics, and cloud-based technology solutions.\n\nThe company is committed to creating an inclusive, professional, innovative, and employee-friendly workplace. This HR Policy establishes the general rules, procedures, benefits, responsibilities, and expectations applicable to employees of CognivexaAI Solutions Pvt. Ltd.\n\n\n2. PURPOSE OF THE HR POLICY\n\nThe purpose of this policy is to provide employees with clear information regarding:\n\n- Working hours and attendance\n- Leave and 

### LANGCHAIN DOCUMENT

Langchain processes everything in form of documents

DOCUMENTS :
PAGE CONTENT -- the actual data
METADATA -- extra information about the data

In [67]:
len(documents)

1

In [68]:
documents[0].page_content
#print(documents[0].page_content)

"COGNIVEXA AI SOLUTIONS PVT. LTD.\nHUMAN RESOURCES POLICY\n\nDocument Title: Employee Human Resources Policy\nCompany: CognivexaAI Solutions Pvt. Ltd.\nDocument Version: 1.0\nEffective Date: 07 August 2026\nDocument Owner: Human Resources Department\n\n\n1. COMPANY OVERVIEW\n\nCognivexaAI Solutions Pvt. Ltd. is a technology company focused on Artificial Intelligence, Machine Learning, Generative AI, software engineering, data analytics, and cloud-based technology solutions.\n\nThe company is committed to creating an inclusive, professional, innovative, and employee-friendly workplace. This HR Policy establishes the general rules, procedures, benefits, responsibilities, and expectations applicable to employees of CognivexaAI Solutions Pvt. Ltd.\n\n\n2. PURPOSE OF THE HR POLICY\n\nThe purpose of this policy is to provide employees with clear information regarding:\n\n- Working hours and attendance\n- Leave and holidays\n- Remote and hybrid work\n- Employee conduct\n- Compensation and pay

In [69]:
documents[0].metadata

{'source': 'data/hr_policy.txt'}

In [70]:
print(f"Total characters in document: {len(documents[0].page_content)}")
# We gwet this information beacuse based on this we decied how to beake into pieces

Total characters in document: 19677


SPLITTING OUR DATA

In [71]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, # Maximum size of each text chunk
    chunk_overlap = 100 # Number of characters shared between consecutive chunks
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COGNIVEXA AI SOLUTIONS PVT. LTD.\nHUMAN RESOURCES POLICY\n\nDocument Title: Employee Human Resources Policy\nCompany: CognivexaAI Solutions Pvt. Ltd.\nDocument Version: 1.0\nEffective Date: 07 August 2026\nDocument Owner: Human Resources Department\n\n\n1. COMPANY OVERVIEW\n\nCognivexaAI Solutions Pvt. Ltd. is a technology company focused on Artificial Intelligence, Machine Learning, Generative AI, software engineering, data analytics, and cloud-based technology solutions.\n\nThe company is committed to creating an inclusive, professional, innovative, and employee-friendly workplace. This HR Policy establishes the general rules, procedures, benefits, responsibilities, and expectations applicable to employees of CognivexaAI Solutions Pvt. Ltd.\n\n\n2. PURPOSE OF THE HR POLICY\n\nThe purpose of this policy is to provide employees with clear information regarding:'), Document(metadata={'source': 'data/hr_policy.txt'}, page

In [72]:
len(chunks)

22

NOW EACH SPLITED CHUNK IS A DOCUMENT - page content and metadata

In [73]:
print(chunks[0])

page_content='COGNIVEXA AI SOLUTIONS PVT. LTD.
HUMAN RESOURCES POLICY

Document Title: Employee Human Resources Policy
Company: CognivexaAI Solutions Pvt. Ltd.
Document Version: 1.0
Effective Date: 07 August 2026
Document Owner: Human Resources Department


1. COMPANY OVERVIEW

CognivexaAI Solutions Pvt. Ltd. is a technology company focused on Artificial Intelligence, Machine Learning, Generative AI, software engineering, data analytics, and cloud-based technology solutions.

The company is committed to creating an inclusive, professional, innovative, and employee-friendly workplace. This HR Policy establishes the general rules, procedures, benefits, responsibilities, and expectations applicable to employees of CognivexaAI Solutions Pvt. Ltd.


2. PURPOSE OF THE HR POLICY

The purpose of this policy is to provide employees with clear information regarding:' metadata={'source': 'data/hr_policy.txt'}


In [74]:
print(chunks[2])

page_content='The standard lunch break is from 1:00 PM to 2:00 PM.

Employees are expected to report to work on time and remain available during their scheduled working hours.

Employees who expect to arrive late must inform their reporting manager as early as possible.


4. ATTENDANCE POLICY

Employees must maintain regular and punctual attendance.

Attendance may be recorded through the company's attendance management system, employee portal, biometric system, or another system designated by the HR Department.

Employees must accurately record their working hours.

Repeated unauthorized absence, late attendance, or early departure may result in corrective action.

If an employee is unable to attend work due to an emergency or unexpected situation, the employee should notify the reporting manager and HR Department as soon as reasonably possible.


5. REMOTE AND HYBRID WORK POLICY' metadata={'source': 'data/hr_policy.txt'}


In [75]:
print(chunks[2].page_content)

The standard lunch break is from 1:00 PM to 2:00 PM.

Employees are expected to report to work on time and remain available during their scheduled working hours.

Employees who expect to arrive late must inform their reporting manager as early as possible.


4. ATTENDANCE POLICY

Employees must maintain regular and punctual attendance.

Attendance may be recorded through the company's attendance management system, employee portal, biometric system, or another system designated by the HR Department.

Employees must accurately record their working hours.

Repeated unauthorized absence, late attendance, or early departure may result in corrective action.

If an employee is unable to attend work due to an emergency or unexpected situation, the employee should notify the reporting manager and HR Department as soon as reasonably possible.


5. REMOTE AND HYBRID WORK POLICY


### EMBEDD OUR DATA
Text into numbers

In [76]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v4")

print("EMB MODEL ID READY THE NAME IS ",embeddings_model.model_name)

EMB MODEL ID READY THE NAME IS  jina-embeddings-v4


# STORE DATA IN VECTOR DB

In [77]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("CHUNKS ARE STORED ", vector_store.index.ntotal)

CHUNKS ARE STORED  22


In [81]:
test_query = "How many sick leaves employee get"

## This is called as similarity search

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")

for i,match in enumerate(top_matches, start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employee get

--- Match 1 ---
Employees should inform their reporting manager as soon as possible when taking Sick Leave.

For extended periods of absence, the HR Department may request appropriate medical documentation.

Employees should not work while seriously ill if doing so could negatively affect their health or workplace safety.


9. EARNED LEAVE

Employees may receive 18 days of Earned Leave per calendar year, subject to eligibility and applicable company rules.

Earned Leave should generally be planned in advance and approved by the reporting manager.

Employees are encouraged to plan longer periods of leave sufficiently in advance to minimize disruption to business operations.

The carry-forward or encashment of unused Earned Leave will be governed by the company's applicable leave rules.


10. PUBLIC HOLIDAYS

CognivexaAI Solutions Pvt. Ltd. observes public and national holidays according to the company's annual holiday calendar.

--- Match 2 ---


TOOL

In [95]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3}) # return 3 relevence 
def search_hr_policy(question:str)->str:
    """Search the HR policy document for information abput leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL

LLM

In [84]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.7 # creativity of the model
)

llm.model_name

'openai/gpt-oss-120b'

In [85]:
test_response = llm.invoke("Hey is learning Rag hard? answer in 1 line")
test_response.content

'RAG can be initially challenging, but with clear resources and practice it’s quite manageable.'

AI AGENT

3 THINGS

1. LLM - BRAIN
2. TOOL - SUPER POWER
3. MEMORY - NO MEMORY

In [86]:
from langchain.agents import create_agent

In [96]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt="""
You are the official HR Assistant for CognivexaAI Solutions Pvt. Ltd.

Your job is to answer employee questions using the company's HR Policy stored in the knowledge base.

RULES:

1. Always use the search_hr_policy tool before answering any HR policy-related question.

2. Use only the information returned by the search_hr_policy tool to answer questions about:
   - Working hours and attendance
   - Leave and holidays
   - Remote and hybrid work
   - Compensation and salary
   - Payroll
   - Performance management
   - Performance Improvement Plans
   - Promotions
   - Training and development
   - Code of conduct
   - Anti-harassment
   - Equal opportunity
   - Conflict of interest
   - Confidentiality
   - Information security
   - Company devices and assets
   - Acceptable technology usage
   - Artificial Intelligence and Generative AI usage
   - Social media
   - Expense reimbursement
   - Business travel
   - Employee grievances
   - Whistleblower policy
   - Workplace health and safety
   - Professional communication
   - Employee privacy
   - Resignation
   - Notice period
   - Exit process
   - Termination
   - Final settlement
   - Policy violations
   - Policy review
   - HR contact information

3. Never guess, assume, or invent an HR policy, rule, benefit, salary detail, leave entitlement, or procedure.

4. If the search results do not contain enough information to answer the question, say:
   "I could not find this information in the CognivexaAI Solutions Pvt. Ltd. HR Policy."

5. Give answers based on the retrieved policy content and preserve the meaning of the original policy.

6. When possible, mention the relevant policy section in your answer.
   Example:
   "According to Section 7, Casual Leave..."

7. If the policy contains a specific number, date, duration, time, or requirement, provide it accurately.

8. Do not combine information from unrelated sections unless it is necessary to answer the employee's question.

9. Keep answers clear, professional, and concise.

10. If the user asks a general question unrelated to CognivexaAI Solutions Pvt. Ltd. HR policies, answer normally only when you have sufficient information. Do not present general information as company policy.

11. If a question requires information that is not present in the HR Policy, clearly state that the information is not available in the policy instead of making an assumption.

Your priority is:
HR Policy Retrieval → Accurate Answer → No Hallucination
"""
)

print("HR assistand agent is ready to answer question!")

HR assistand agent is ready to answer question!


In [98]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke(
        {
        "messages": [
            {
                "role": "user",
                "content": question
            }]
        }
    )
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [99]:
response = hr_assistant.invoke(
        {
        "messages": [
            {
                "role": "user",
                "content": "tell me about leave policies how to apply for leave"
            }]
        }
    )

In [100]:
response

{'messages': [HumanMessage(content='tell me about leave policies how to apply for leave', additional_kwargs={}, response_metadata={}, id='ec79fdb2-8d02-4017-9233-975537663439'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to search HR policy for leave policies and application process. Use search_hr_policy tool.', 'tool_calls': [{'id': 'fc_69070fe3-bff1-4532-87d4-02b2a110fed2', 'function': {'arguments': '{"question":"leave policies how to apply for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 683, 'total_tokens': 737, 'completion_time': 0.114365784, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.118348346, 'prompt_tokens_details': None, 'queue_time': 0.396128129, 'total_time': 0.23271413}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 

SYSTEM MESSAGE - HR ASSISTANT

HUMAN MESSAGE - TELL ME ABOUT POLICIES

AI MESSAGE - HEY THESE ARE THE POLICIES

In [102]:
response["messages"][-2].content

"Leave requests should normally be submitted through the company's leave management system.\n\nEmployees should obtain approval from their reporting manager before taking planned leave.\n\nEmergency or medical leave may be communicated to the reporting manager as soon as possible when advance approval is not practical.\n\n\n7. CASUAL LEAVE\n\nCasual Leave may be used for personal requirements, short-term personal matters, or unexpected situations.\n\nEmployees are generally eligible for 12 days of Casual Leave per calendar year.\n\nCasual Leave should normally be requested at least one working day in advance.\n\nUnused Casual Leave may not automatically carry forward to the following year unless specifically permitted by company policy.\n\n\n8. SICK LEAVE\n\nSick Leave may be used when an employee is unable to work due to illness or a medical condition.\n\nEmployees should inform their reporting manager as soon as possible when taking Sick Leave.\n\nEmployees should inform their report

In [104]:
print(response["messages"][-1].content)

**Leave Policies Overview & How to Apply**

1. **General Procedure**  
   - All leave requests must be submitted through the company’s **leave management system**.  
   - Obtain approval from your **reporting manager** before taking any planned leave.  
   - For emergency or medical leave where advance approval isn’t possible, inform your reporting manager **as soon as possible** (Section 7‑9).

2. **Types of Leave & Entitlements**  

   | Leave Type | Entitlement | Key Requirements |
   |------------|------------|-------------------|
   | **Casual Leave** | 12 days per calendar year | Request at least **one working day in advance** (Section 7). Unused days generally do not carry forward unless expressly allowed. |
   | **Sick Leave** | (not specified in excerpt) | Inform your reporting manager **as soon as possible**. For extended absences, HR may request medical documentation (Section 8). |
   | **Earned Leave** | 18 days per calendar year | Plan **in advance** and obtain manager app